In [4]:
import numpy as np
import pandas as pd
import json
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
import joblib

# 1. Carregar os dados do seu arquivo JSON do Firebase
with open('consumo-conciente.json', 'r') as f:
    dados_firebase = json.load(f)

# Pegar os dados que estão dentro do nó principal "leituras"
if "leituras" in dados_firebase:
    dados_raiz = dados_firebase["leituras"]
else:
    dados_raiz = dados_firebase

# Lista para armazenar as linhas achatadas
linhas_achatadas = []

# Varrer a árvore do JSON: Data -> Hora -> Atributos Elétricos
for data, horas in dados_raiz.items():
    if isinstance(horas, dict):
        for hora, atributos in horas.items():
            if isinstance(atributos, dict):
                # Se o campo se chamar 'consumo_acumulado', mapeia para 'consumo'
                if 'consumo_acumulado' in atributos and 'consumo' not in atributos:
                    atributos['consumo'] = atributos['consumo_acumulado']

                # Armazena uma cópia dos dados elétricos
                linhas_achatadas.append(atributos)

# Criar o DataFrame com as linhas planas
df = pd.DataFrame(linhas_achatadas)

# Definir as colunas monitoradas e filtrar dados nulos
COLUNAS_MONITORADAS = ['consumo', 'corrente', 'potencia', 'voltagem']
df_filtrado = df[COLUNAS_MONITORADAS].dropna()

print(f"Total de amostras prontas para o treino: {len(df_filtrado)}")
print("Exemplo dos dados mapeados:\n", df_filtrado.head())

# 2. Normalização dos dados (0 a 1)
scaler = MinMaxScaler(feature_range=(0, 1))
dados_normalizados = scaler.fit_transform(df_filtrado)

# Salvar o scaler elétrico atualizado
joblib.dump(scaler, 'scaler_energia.pkl')

# 3. Criação da Janela Temporal (60 passos)
X, y = [], []
JANELA = 60
INDICE_CONSUMO = COLUNAS_MONITORADAS.index('consumo')

for i in range(JANELA, len(dados_normalizados)):
    X.append(dados_normalizados[i-JANELA:i])       # As 60 leituras anteriores das 4 features
    y.append(dados_normalizados[i, INDICE_CONSUMO]) # O consumo do próximo passo (Target)

X, y = np.array(X), np.array(y)

# 4. Construção e Treinamento do Modelo LSTM
model = Sequential([
    LSTM(50, return_sequences=True, input_shape=(X.shape[1], X.shape[2])),
    Dropout(0.2),
    LSTM(50, return_sequences=False),
    Dropout(0.2),
    Dense(25),
    Dense(1) # Saída: Consumo Previsto
])

model.compile(optimizer='adam', loss='mean_squared_error')
model.fit(X, y, batch_size=32, epochs=20)

# Salvar os arquivos finais da IA
model.save('modelo_lstm_energia.keras')
print("Modelo treinado focado em CONSUMO salvo com sucesso!")

Total de amostras prontas para o treino: 2159
Exemplo dos dados mapeados:
    consumo  corrente  potencia   voltagem
0   12.450     1.200     153.0  127.50000
1    0.007     0.120       8.8  133.60001
2    0.007     0.118       8.6  133.70000
3    0.008     0.134      10.2  133.89999
4    0.008     0.108       7.8  133.89999
Epoch 1/20


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


66/66 ━━━━━━━━━━━━━━━━━━━━ 6s 45ms/step - loss: 4.8635e-04
Epoch 2/20
66/66 ━━━━━━━━━━━━━━━━━━━━ 4s 63ms/step - loss: 4.7483e-05
Epoch 3/20
66/66 ━━━━━━━━━━━━━━━━━━━━ 3s 44ms/step - loss: 3.3039e-05
Epoch 4/20
66/66 ━━━━━━━━━━━━━━━━━━━━ 5s 45ms/step - loss: 2.6480e-05
Epoch 5/20
66/66 ━━━━━━━━━━━━━━━━━━━━ 4s 63ms/step - loss: 2.2666e-05
Epoch 6/20
66/66 ━━━━━━━━━━━━━━━━━━━━ 3s 45ms/step - loss: 1.9350e-05
Epoch 7/20
66/66 ━━━━━━━━━━━━━━━━━━━━ 3s 45ms/step - loss: 1.7546e-05
Epoch 8/20
66/66 ━━━━━━━━━━━━━━━━━━━━ 6s 55ms/step - loss: 1.4757e-05
Epoch 9/20
66/66 ━━━━━━━━━━━━━━━━━━━━ 4s 45ms/step - loss: 1.3005e-05
Epoch 10/20
66/66 ━━━━━━━━━━━━━━━━━━━━ 3s 45ms/step - loss: 1.2103e-05
Epoch 11/20
66/66 ━━━━━━━━━━━━━━━━━━━━ 3s 45ms/step - loss: 9.9863e-06
Epoch 12/20
66/66 ━━━━━━━━━━━━━━━━━━━━ 5s 46ms/step - loss: 8.7577e-06
Epoch 13/20
66/66 ━━━━━━━━━━━━━━━━━━━━ 5s 45ms/step - loss: 6.8119e-06
Epoch 14/20
66/66 ━━━━━━━━━━━━━━━━━━━━ 4s 54ms/step - loss: 6.0016e-06
Epoch 15/20
66/66 ━━━━━━━━